# StockSense — Predicting Daily Stock Movement with Machine Learning

**Goal:** Build a complete ML pipeline that predicts whether a stock will close *higher or lower* the next trading day.

**Why this problem?**
I chose this problem because stock price movement is one of the most studied, most difficult prediction tasks in ML. It's noisy, non-stationary, and full of many, common pitfalls with data. This makes it an excellent project for learning: every concept I studied here — data leakage, feature engineering, class imbalance, overfitting — is something engineers genuinely wrestle with in production systems.

**End deliverable:** A single, fully documented Jupyter notebook — raw data in, trained models and evaluation charts out.

## 0. Imports

All dependencies are declared at the top of the notebook in a single cell. This is standard practice in data science notebooks. Full requirements can be found in the requirments.txt file in the project root directory.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as npy
import matplotlib.pyplot as plt
import seaborn as sbn

## 1.1 — Acquiring the Data

**What I'm doing:** Downloading 5 years of daily OHLCV data for Apple Inc. (AAPL) from Yahoo Finance via the `yfinance` API.

**Why Apple (AAPL)?**
- Highly liquid — prices reflect genuine market activity
- Has 5 years of history that gives ~1,260 trading days, which is enough for meaningful ML without being unwieldy
- No survivorship bias concern for a single ticker used as a learning exercise
- Free and accessible via `yfinance`

**OHLCV defined:**
| Column | Meaning |
|--------|---------|
| Open | Price at market open |
| High | Intraday highest price |
| Low | Intraday lowest price |
| Close | Price at market close |
| Volume | Number of shares traded |
| Adj Close | Close adjusted for dividends and stock splits |

> [!NOTE] Data Clarification:
> - I'm using raw Close rather than Adj Close for simplicity. For a production system analysing longer time horizons or dividend-heavy stocks, Adj Close would
> be the best choice, as it removes price discontinuities caused by dividends and stock splits.

In [ ]:
df = yf.download('AAPL', period='5y')

### Inspecting the data — never skip this

Before any transformation, I run three standard inspection calls, called a **sanity check**. You will waste hours debugging a model that has a broken input.

- `df.head()` — confirm the column names, index type (should be `DatetimeIndex`), and that values look reasonable
- `df.info()` — check dtypes and the count of non-null entries per column. Any column with missing values will show a lower count than the total rows
- `df.describe()` — check the statistical summary: min, max, mean, std. A negative minimum Volume, for example, would immediately flag a data issue

> 🔑 **What I'm watching for:**
> - Missing values (`NaN`s) — yfinance data is generally clean, but market holidays and early data can introduce gaps
> - Outliers in data or corrupted data is very important to confirm before doing features and splitting data.

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

### The train / validation / test split — the most important concept in applied ML

Before I engineer a single feature, I need to split the data. This ordering is non-negotiable.

**The three sets:**
| Set | Purpose | Approximate size |
|-----|---------|-----------------|
| Train | The model learns from this | ~70% (first ~880 rows) |
| Validation | I tune and compare models here | ~15% (next ~190 rows) |
| Test | Final honest evaluation — looked at *once*, at the very end | ~15% (last ~190 rows) |

---

> 🔑 **Decision point — why I split *chronologically*, not randomly:**
> This is time-series data. Each row represents a day that happens *after* all previous rows. If you randomly shuffle the data before splitting — which is fine for most ML datasets — you will have rows from 2024 in your training set and rows from 2019 in your test set. The model effectively "sees the future" during training. This is called **data leakage**, and it produces test set performance numbers that look good in the notebook but collapse completely in production.
    - must ensure that data is also split properly so no test or validation data is leaked into the training data; which also cause "seeing into the future".

> 🔑 **Decision point — why I have a *separate* validation set at all:**
> Every time you look at your test set performance and adjust your model based on it, you're subtly overfitting to the test set. It ceases to be a true held-out set. The validation set exists to absorb all of your tuning decisions. You can evaluate on it as many times as you like. The test set is touched exactly once at the end and whatever the reported result, thats what you get, with no further changes.

In [ ]:
data_size = len(df)
# 70% training data, 15% validation data, 15% test data put into new data frames
train_df = df.iloc[:int(data_size * 0.7)]
val_df = df.iloc[int(data_size * 0.7):int(data_size * 0.85)]
test_df = df.iloc[int(data_size * 0.85):]

In [ ]:
# Ensure data was split correctly
print(f'Train set shape: {train_df.shape}')
print(f'Val set shape:   {val_df.shape}')
print(f'Test set shape:  {test_df.shape}')
if train_df.shape[0] + val_df.shape[0] + test_df.shape[0] == data_size:
    print('Data split correctly!')
else:
    print('Data was not split correctly!')

In [ ]:
short_window = 10
long_window = 50
# Perform a Simple Moving Average of closing over last n days
# short term avgs > long term avgs = buy signal
df['SMA10'] = df['Close'].rolling(window=short_window).mean()
df['SMA50'] = df['Close'].rolling(window=long_window).mean()
df[['Close', 'SMA10', 'SMA50']].head(50)

In [ ]:
fast_decay = 12
slow_decay = 26
# Performs a Exponential Moving Average
# Similar to SMA but gives more weight to recent prices
df['EMA12'] = df['Close'].ewm(span=fast_decay).mean()
df['EMA26'] = df['Close'].ewm(span=slow_decay).mean()
# EMA_12 > EMA_26 => Bullish; EMA_12 < EMA_26 => Bearish
df['MACD'] = df['EMA12'] - df['EMA26']
df[['Close', 'EMA12', 'EMA26', 'MACD']].head(50)

In [ ]:
# Finding daily change in price
daily_change = df['Close'].diff()
rsi_window = 14
# Separating daily_change col into two separate col's: gain/loss
gain = daily_change.where(daily_change > 0, 0)
loss = -daily_change.where(daily_change < 0, 0)
# Compute rolling avg's
avg_gain = gain.rolling(window=rsi_window).mean()
avg_loss = loss.rolling(window=rsi_window).mean()
# Compute RS
rs = avg_gain / avg_loss
# Compute RSI so values are 0 -> 100 and not 0 -> inf
df['RSI'] = 100 - (100 / (1 + rs))

In [ ]:
# Verify 0 <= x <= 100
df['RSI'].describe()

In [ ]:
# Visualization: verifying values are generally between 20-80
df['RSI'].plot()